# 00 - Data Cleaning
This notebook loads the raw Retention and BoB datasets, cleans them, and saves the results to `data/processed/`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from config import (
    RETENTION_FILE, BOB_FILE, PROCESSED_DATA_DIR,
    COMPANY_SIZE_MAP, COMPANY_SIZE_NUMERIC_MAP,
    CHURN_STATUS, CHURN_REASON_KEYWORDS
)
from src.util import save_dataframe

## Load raw data

In [ ]:
# load retention data
retention = pd.read_csv(RETENTION_FILE, encoding='latin-1')
print(f'Retention: {len(retention):,} rows, {len(retention.columns)} columns')
print(f'Unique Case IDs: {retention["Case ID"].nunique():,}')

# load BoB data
bob = pd.read_excel(BOB_FILE)
print(f'\nBoB: {len(bob):,} rows, {len(bob.columns)} columns')
print(f'Unique Accounts: {bob["account_number"].nunique():,}')

## Clean Retention data

In [ ]:
# drop columns that are 100% null
drop_cols = [c for c in retention.columns if retention[c].isnull().all()]
if drop_cols:
    retention = retention.drop(columns=drop_cols)
    print(f'Dropped 100% null columns: {drop_cols}')

# remove exact duplicates
n_before = len(retention)
retention = retention.drop_duplicates()
print(f'Removed {n_before - len(retention):,} exact duplicates')

# remove Case ID duplicates (keep first)
n_before = len(retention)
retention = retention.drop_duplicates(subset='Case ID', keep='first')
print(f'Removed {n_before - len(retention):,} Case ID duplicates')

print(f'\nAfter dedup: {len(retention):,} rows')

In [ ]:
# filter to resolved cases only and create target variable
resolved_statuses = ['Customer Lost', 'Customer Saved']
retention = retention[retention['Resolution Status'].isin(resolved_statuses)].copy()
retention['is_churned'] = (retention['Resolution Status'] == CHURN_STATUS).astype(int)

churn_rate = retention['is_churned'].mean()
print(f'Resolved cases: {len(retention):,}')
print(f'Customer Lost: {retention["is_churned"].sum():,} ({churn_rate:.1%})')
print(f'Customer Saved: {(~retention["is_churned"].astype(bool)).sum():,} ({1-churn_rate:.1%})')

In [ ]:
# fix CompanySize (Excel date corruption)
if 'CompanySize' in retention.columns:
    print('Before:', retention['CompanySize'].value_counts().to_dict())
    retention['CompanySize'] = retention['CompanySize'].map(COMPANY_SIZE_MAP).fillna(retention['CompanySize'])
    retention['CompanySize_numeric'] = retention['CompanySize'].map(COMPANY_SIZE_NUMERIC_MAP)
    print('After:', retention['CompanySize'].value_counts().to_dict())

In [ ]:
# fix Customer Tier inconsistency
if 'Customer Tier' in retention.columns:
    retention['Customer Tier'] = retention['Customer Tier'].str.strip().replace({'Platinum +': 'Platinum+'})
    print('Customer Tier values:', retention['Customer Tier'].value_counts().to_dict())

In [ ]:
# clip negative New VAN values to 0
if 'New VAN' in retention.columns:
    n_neg = (retention['New VAN'] < 0).sum()
    if n_neg > 0:
        retention['New VAN'] = retention['New VAN'].clip(lower=0)
        print(f'Clipped {n_neg} negative values in New VAN to 0')

# flag zero VAN rows
if 'VAN' in retention.columns:
    retention['is_zero_van'] = (retention['VAN'] == 0).astype(int)
    print(f'Zero VAN rows: {retention["is_zero_van"].sum():,} ({retention["is_zero_van"].mean():.1%})')

In [ ]:
# categorize churn reasons from Case Title
def categorize_churn_reason(title):
    if pd.isna(title):
        return 'Unknown'
    title_lower = str(title).lower()
    for reason, keywords in CHURN_REASON_KEYWORDS.items():
        if any(kw in title_lower for kw in keywords):
            return reason
    return 'Other'

retention['churn_reason_group'] = retention['Case Title'].apply(categorize_churn_reason)
print('Churn reasons:', retention['churn_reason_group'].value_counts().to_dict())

In [ ]:
# handle nulls
# categorical columns -> fill with 'Unknown' or similar
cat_fill = {
    'Risk': 'Unknown', 'Pull Type': 'None', 'Customer Tier': 'Unknown',
    'CompanySize': 'Unknown', 'Case Origin': 'Unknown', 'Branch': 'Unknown'
}
for col, fill_val in cat_fill.items():
    if col in retention.columns and retention[col].isnull().sum() > 0:
        null_pct = retention[col].isnull().mean()
        if null_pct > 0.1:
            retention[f'{col}_missing'] = retention[col].isnull().astype(int)
        retention[col] = retention[col].fillna(fill_val)
        print(f'{col}: filled nulls with "{fill_val}"')

# numeric columns -> fill with 0 for repair/overdue, median for others
for col in ['Number Of Repair Cases', 'Number of OverdueServices']:
    if col in retention.columns and retention[col].isnull().sum() > 0:
        retention[col] = retention[col].fillna(0)
        print(f'{col}: filled nulls with 0')

numeric_cols = retention.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if retention[col].isnull().sum() > 0:
        median_val = retention[col].median()
        retention[col] = retention[col].fillna(median_val)
        print(f'{col}: filled nulls with median ({median_val:.2f})')

print(f'\nRemaining nulls: {retention.isnull().sum().sum()}')

## Clean BoB data

In [ ]:
# remove duplicates
n_before = len(bob)
bob = bob.drop_duplicates()
print(f'Removed {n_before - len(bob):,} exact duplicates')

# filter to active records only
n_before = len(bob)
bob = bob[bob['system_status'] == 'Active'].copy()
print(f'Filtered to Active: {n_before:,} -> {len(bob):,}')

# fix empty company_sizing
if 'company_sizing' in bob.columns:
    bob['company_sizing'] = bob['company_sizing'].replace('', np.nan)

print(f'\nBoB cleaned: {len(bob):,} rows, {len(bob.columns)} columns')

## Save cleaned data

In [ ]:
import os
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

save_dataframe(retention, os.path.join(PROCESSED_DATA_DIR, 'retention_cleaned.csv'), 'cleaned retention')
save_dataframe(bob, os.path.join(PROCESSED_DATA_DIR, 'bob_cleaned.csv'), 'cleaned BoB')

print(f'\nRetention final shape: {retention.shape}')
print(f'BoB final shape: {bob.shape}')